In [1]:
# ============================================================
# MARKETPLACE REVIEW INTELLIGENCE
# Notebook 04 — NLP e Análise de Sentimento
# ============================================================
# Objetivo: Analisar o sentimento dos reviews usando léxico
# manual calibrado para o domínio cosmético PT-BR.
#
# Entregas:
# - sentimento_score  → float entre -1 e 1
# - classificacao     → Positivo / Neutro / Negativo
# - palavras-chave    → top palavras por faixa de rating
# ============================================================

import pandas as pd
import re
import nltk
from collections import Counter
from pathlib import Path

# Caminhos do projeto
ROOT = Path().resolve().parent
PROCESSED_PATH = ROOT / "data" / "processed"

# Carrega o dataset enriquecido do Notebook 03
df = pd.read_csv(PROCESSED_PATH / "reviews_enriquecidos.csv", encoding="utf-8-sig")
df["date"] = pd.to_datetime(df["date"])

print(f"✅ Dataset carregado: {df.shape[0]:,} registros")
print(f"📋 Colunas: {list(df.columns)}")

✅ Dataset carregado: 202,785 registros
📋 Colunas: ['date', 'rating', 'content', 'product_url', 'qtd_palavras', 'review_curto', 'content_original', 'produto_id', 'slug', 'nome_produto', 'categoria']


In [2]:
# ============================================================
# Download dos recursos necessários do NLTK
# ============================================================

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Negadores definidos ANTES das stopwords
# para garantir que sejam protegidos
NEGADORES = {"nao", "nunca", "jamais", "nem", "tampouco", "sequer"}

# Stopwords em português
STOPWORDS_PT = set(stopwords.words("portuguese"))

# Stopwords adicionais do domínio cosmético
STOPWORDS_EXTRA = {
    "ml", "kg", "gr", "g", "l", "un", "und", "kit",
    "produto", "produtos", "item", "itens", "compra",
    "comprei", "recebi", "chegou", "entrega", "vendedor",
    "mercado", "livre", "prazo", "embalagem", "original"
}

STOPWORDS_PT.update(STOPWORDS_EXTRA)

# ✅ CORREÇÃO CRÍTICA — remove negadores das stopwords
# Sem isso "não", "nunca", "jamais" eram descartados
# e toda negação do texto era invisível para o NLP
STOPWORDS_PT -= NEGADORES

print(f"✅ NLTK configurado")
print(f"📚 Total de stopwords: {len(STOPWORDS_PT)}")
print(f"🔄 Negadores protegidos: {NEGADORES}")
print(f"\n✅ Verificação — negadores NÃO estão nas stopwords:")
for neg in NEGADORES:
    status = "❌ PROBLEMA" if neg in STOPWORDS_PT else "✅ Protegido"
    print(f"   '{neg}' → {status}")

✅ NLTK configurado
📚 Total de stopwords: 229
🔄 Negadores protegidos: {'nem', 'sequer', 'nao', 'jamais', 'tampouco', 'nunca'}

✅ Verificação — negadores NÃO estão nas stopwords:
   'nem' → ✅ Protegido
   'sequer' → ✅ Protegido
   'nao' → ✅ Protegido
   'jamais' → ✅ Protegido
   'tampouco' → ✅ Protegido
   'nunca' → ✅ Protegido


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\edigu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\edigu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\edigu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [6]:
# ============================================================
# Notebook 04 — Bloco 3 CORREÇÃO FINAL
# ============================================================
# Correção: normalização de acentos DENTRO da tokenização
# "não" → "nao" antes do filtro de stopwords
# ============================================================

import unicodedata

def remover_acentos(texto):
    """
    Converte caracteres acentuados para ASCII.
    "não" → "nao", "ótimo" → "otimo"
    """
    return unicodedata.normalize("NFKD", texto)\
           .encode("ASCII", "ignore")\
           .decode("ASCII")

def preprocessar_texto(texto):
    """
    Tokeniza o texto e remove stopwords.
    PRESERVA negadores mesmo sendo stopwords.
    Normaliza acentos ANTES do filtro — garante que
    "não" → "nao" e seja reconhecido como negador.
    """
    if pd.isna(texto) or str(texto).strip() == "":
        return []

    # Normaliza acentos antes de tudo
    texto_normalizado = remover_acentos(str(texto).lower())

    # Tokeniza
    tokens = word_tokenize(texto_normalizado, language="portuguese")

    # Remove stopwords MAS preserva negadores
    tokens_limpos = [
        t for t in tokens
        if t.isalpha()
        and len(t) > 1
        and (t not in STOPWORDS_PT or t in NEGADORES)
    ]

    return tokens_limpos

# Aplica no dataset
df["tokens"] = df["content"].apply(preprocessar_texto)

# Diagnóstico
media_tokens = df["tokens"].apply(len).mean()
zero_tokens  = (df["tokens"].apply(len) == 0).sum()

print(f"✅ Tokenização corrigida com normalização de acentos")
print(f"📊 Média de tokens por review: {media_tokens:.1f}")
print(f"⚠️  Reviews sem tokens: {zero_tokens:,}")

print(f"\n🔍 Verificação de negadores nos tokens:")
exemplos_negacao = df[
    df["content"].str.contains("não|nunca|jamais", case=False, na=False)
][["content", "tokens"]].head(6)

for _, row in exemplos_negacao.iterrows():
    print(f"\n  Original : {row['content'][:70]}")
    print(f"  Tokens   : {row['tokens']}")

✅ Tokenização corrigida com normalização de acentos
📊 Média de tokens por review: 5.8
⚠️  Reviews sem tokens: 868

🔍 Verificação de negadores nos tokens:

  Original : ótimo recomendo e já usei a fragrância é bem fraquinha eu que quase tu
  Tokens   : ['otimo', 'recomendo', 'ja', 'usei', 'fragrancia', 'bem', 'fraquinha', 'quase', 'tudo', 'alergia', 'nao', 'nem', 'reacao']

  Original : se fizer direitinho o cabelo fica muito bom de se manter alinhado, eu 
  Tokens   : ['fizer', 'direitinho', 'cabelo', 'fica', 'bom', 'manter', 'alinhado', 'gosto', 'desse', 'deixa', 'cabelo', 'molinho', 'naturalidade', 'particularmente', 'nao', 'gosto', 'desses', 'alizar', 'fica', 'alizado', 'forcado', 'tipo', 'artificial', 'espigado']

  Original : este shampoo é maravilhoso. eu uso ele a um ano e não fico sem.
  Tokens   : ['shampoo', 'maravilhoso', 'uso', 'ano', 'nao', 'fico']

  Original : ainda não usei, breve voltarei p relatar.
  Tokens   : ['ainda', 'nao', 'usei', 'breve', 'voltarei', 'relatar']


In [9]:
# mostrar as colunas content e tokens lado a lado
df[["content", "tokens"]].tail(10)

,content,tokens
202775,"para meu cabelo não deu certo ,pq já é seco fi...","[cabelo, nao, deu, certo, pq, ja, seco, ficou,..."
202776,não resolve nada para oleosidade.,"[nao, resolve, nada, oleosidade]"
202777,excelente produto.,[excelente]
202778,produto excelente.,[excelente]
202779,super bom.,"[super, bom]"
202780,não é muito cheiroso e não deixa o cabelo maci...,"[nao, cheiroso, nao, deixa, cabelo, macio, esp..."
202781,muito bom ! recomendo.,"[bom, recomendo]"
202782,maravilhoso! comprarei mais vezes.,"[maravilhoso, comprarei, vezes]"
202783,não é original!. comprei um menor na loja de c...,"[nao, menor, loja, cosmeticos, antes, comprar,..."
202784,"é bom, mais já usei produtos bem melhores e ma...","[bom, ja, usei, bem, melhores, baratos]"


In [10]:
# ============================================================
# MARKETPLACE REVIEW INTELLIGENCE
# Notebook 04 — Bloco 4 CORRIGIDO
# ============================================================
# Correções aplicadas:
# 1. Removidas palavras ambíguas que causavam distorção:
#    - "oleoso/oleosa" → descreve cabelo, não o produto
#    - "vale" → muito genérico e ambíguo
#    - "caro" → subjetivo demais
#    - "pesado/pesada" → pode ser positivo em cremes
# 2. Adicionadas palavras específicas do domínio cosmético
#    que estavam faltando no léxico original
# ============================================================

LEXICO_POSITIVO = {
    # Peso +2 — fortemente positivos
    "maravilhoso": 2, "maravilhosa": 2,
    "excelente": 2, "incrivel": 2,
    "perfeito": 2, "perfeita": 2,
    "amei": 2, "adorei": 2,
    "otimo": 2, "otima": 2,
    "fantastico": 2, "fantastica": 2,
    "magico": 2, "magica": 2,
    "surpreendente": 2, "espetacular": 2,
    "sensacional": 2, "melhor": 2,
    "aprovado": 2, "aprovada": 2,
    "top": 2, "recomendo": 2, "recomendado": 2,
    "amando": 2, "lindos": 2, "lindo": 2, "linda": 2,

    # Peso +1 — moderadamente positivos
    "bom": 1, "boa": 1,
    "gostei": 1, "gosto": 1,
    "bonito": 1, "bonita": 1,
    "cheiroso": 1, "cheirosa": 1,
    "hidratado": 1, "hidratada": 1, "hidrata": 1,
    "macio": 1, "macia": 1,
    "brilhoso": 1, "brilhante": 1, "brilho": 1,
    "suave": 1, "leve": 1,
    "funciona": 1, "funcionou": 1,
    "cumpre": 1, "atendeu": 1,
    "satisfeito": 1, "satisfeita": 1,
    "feliz": 1, "contente": 1,
    "rapido": 1, "pratico": 1, "pratica": 1,
    "agradavel": 1, "eficiente": 1, "eficaz": 1,
    "nutritivo": 1, "nutritiva": 1,
    "restaurou": 1, "recuperou": 1,
    "fortaleceu": 1, "cresceu": 1,
    "cheiro": 1, "perfume": 1, "aroma": 1,
    "refrescante": 1, "revitalizou": 1,
}

LEXICO_NEGATIVO = {
    # Peso -2 — fortemente negativos
    "horrivel": -2, "pessimo": -2, "pessima": -2,
    "terrivel": -2, "odiei": -2, "detestei": -2,
    "lixo": -2, "vergonha": -2, "fraude": -2,
    "enganoso": -2, "enganosa": -2,
    "decepcionante": -2, "estragou": -2,
    "queimou": -2, "arrependi": -2,
    "ressecou": -2, "quebrou": -2,
    "horrivel": -2, "nojento": -2, "nojenta": -2,

    # Peso -1 — moderadamente negativos
    # ⚠️ Removidos: "oleoso", "vale", "caro", "pesado"
    # por serem ambíguos no domínio cosmético
    "ruim": -1, "fraco": -1, "fraca": -1,
    "problema": -1, "problemas": -1,
    "demorou": -1, "atrasou": -1,
    "danificou": -1, "irritou": -1,
    "cocou": -1, "resseca": -1,
    "decepcionei": -1, "decepcionou": -1,
    "fraquinho": -1, "fraquinha": -1,
    "regular": -1, "mediano": -1, "mediana": -1,
    "caindo": -1, "queda": -1,
    "ardeu": -1, "arde": -1, "queima": -1,
    "manchou": -1, "alergico": -1, "alergia": -1,
    "cheiro-ruim": -1, "enjoativo": -1, "enjoativa": -1,
    "triste": -1, "frustrado": -1, "frustrada": -1,
}

print(f"✅ Léxico corrigido e carregado")
print(f"📗 Palavras positivas: {len(LEXICO_POSITIVO)}")
print(f"📕 Palavras negativas: {len(LEXICO_NEGATIVO)}")
print(f"🔄 Negadores:         {len(NEGADORES)}")
print(f"\n⚠️  Palavras removidas por ambiguidade:")
print(f"   'oleoso', 'oleosa', 'vale', 'caro', 'pesado', 'pesada'")

✅ Léxico corrigido e carregado
📗 Palavras positivas: 70
📕 Palavras negativas: 51
🔄 Negadores:         6

⚠️  Palavras removidas por ambiguidade:
   'oleoso', 'oleosa', 'vale', 'caro', 'pesado', 'pesada'


In [11]:
# ============================================================
# MARKETPLACE REVIEW INTELLIGENCE
# Notebook 04 — Bloco 5 CORRIGIDO
# ============================================================
# Correções aplicadas:
# 1. Janela de negação ampliada para 2 palavras
#    "não vale a pena" → agora captura "pena" como negada
# 2. Reset do negador apenas após palavra do léxico
#    evita que um negador solto distorça palavras distantes
# ============================================================

def calcular_score(tokens):
    """
    Percorre os tokens do review.
    Aplica pesos do léxico com janela de negação de 2 palavras.
    Normaliza o score pelo total de pesos encontrados.
    Retorna float entre -1 e 1.
    """
    if not tokens:
        return 0.0

    score_bruto      = 0
    total_pesos      = 0
    janela_negacao   = 0  # conta quantas palavras ainda estão na janela

    for token in tokens:

        # Encontrou negador — abre janela de 2 palavras
        if token in NEGADORES:
            janela_negacao = 2
            continue

        # Verifica no léxico positivo
        if token in LEXICO_POSITIVO:
            peso = LEXICO_POSITIVO[token]
            if janela_negacao > 0:
                score_bruto -= peso   # inverte
            else:
                score_bruto += peso
            total_pesos  += abs(peso)
            janela_negacao = 0        # fecha janela após palavra do léxico

        # Verifica no léxico negativo
        elif token in LEXICO_NEGATIVO:
            peso = LEXICO_NEGATIVO[token]
            if janela_negacao > 0:
                score_bruto -= peso   # inverte (dupla negação → positivo)
            else:
                score_bruto += peso
            total_pesos  += abs(peso)
            janela_negacao = 0        # fecha janela após palavra do léxico

        else:
            # Palavra fora do léxico — decrementa janela
            if janela_negacao > 0:
                janela_negacao -= 1

    # Normaliza entre -1 e 1
    if total_pesos == 0:
        return 0.0

    score_normalizado = score_bruto / total_pesos
    return round(max(-1.0, min(1.0, score_normalizado)), 4)

# Aplica no dataset
df["sentimento_score"] = df["tokens"].apply(calcular_score)

print(f"✅ Scores recalculados com negação corrigida")
print(f"\n📊 Distribuição dos scores:")
print(df["sentimento_score"].describe().round(4))

# Verifica casos de negação específicos
print(f"\n🔍 Teste de negação — exemplos críticos:")
casos_teste = df[
    df["content"].str.contains("não gostei|não vale|não recomendo",
    case=False, na=False)
][["content", "tokens", "sentimento_score"]].head(6)

for _, row in casos_teste.iterrows():
    print(f"\n  Texto  : {row['content'][:60]}")
    print(f"  Tokens : {row['tokens']}")
    print(f"  Score  : {row['sentimento_score']}")

✅ Scores recalculados com negação corrigida

📊 Distribuição dos scores:
count    202785.0000
mean          0.7603
std           0.4882
min          -1.0000
25%           1.0000
50%           1.0000
75%           1.0000
max           1.0000
Name: sentimento_score, dtype: float64

🔍 Teste de negação — exemplos críticos:

  Texto  : não gostei não.
  Tokens : ['nao', 'gostei', 'nao']
  Score  : -1.0

  Texto  : não gostei uso essa marca mas não gostei este produto não fu
  Tokens : ['nao', 'gostei', 'uso', 'marca', 'nao', 'gostei', 'nao', 'funcionou', 'tipo', 'cabelo']
  Score  : -1.0

  Texto  : não indico. não gostei produto não matiza o cabelo e olha qu
  Tokens : ['nao', 'indico', 'nao', 'gostei', 'nao', 'matiza', 'cabelo', 'olha', 'deixei', 'minutos', 'cabelo', 'poucas', 'mechas', 'amarelas', 'nem', 'assim', 'matizou', 'hora', 'passa', 'cabelo', 'fica', 'totalmente', 'preto', 'porem', 'hora', 'lava', 'sai', 'absolutamente', 'td', 'nem', 'sombra', 'matizacao', 'joguei', 'dinheiro', 'l

In [12]:
# ============================================================
# MARKETPLACE REVIEW INTELLIGENCE
# Notebook 04 — Bloco 6 CORRIGIDO
# ============================================================
# Correção aplicada: Classificação Híbrida
#
# O rating é a opinião mais direta do cliente.
# Usamos ele como âncora quando há conflito com o score NLP.
#
# Regras da classificação híbrida:
# ┌─────────────────────────────────────────────────────────┐
# │ Rating 4-5 + score >= -0.05  → Positivo                │
# │ Rating 1-2 + score <=  0.05  → Negativo                │
# │ Rating 3                     → confia no score NLP     │
# │ Conflito (rating alto/score negativo) → rating vence   │
# │ Reviews sem palavras no léxico (score 0.0) → rating    │
# └─────────────────────────────────────────────────────────┘
# ============================================================

def classificar_hibrido(row):
    """
    Classificação híbrida: combina score NLP + rating.
    Rating age como âncora quando há conflito ou score neutro.
    """
    score  = row["sentimento_score"]
    rating = row["rating"]

    # Reviews sem palavras no léxico — usa o rating como base
    if score == 0.0:
        if rating >= 4:
            return "Positivo"
        elif rating <= 2:
            return "Negativo"
        else:
            return "Neutro"

    # Rating alto (4-5) — score levemente negativo ainda é Positivo
    if rating >= 4:
        if score >= -0.3:
            return "Positivo"
        else:
            return "Neutro"  # score muito negativo mesmo com rating alto

    # Rating baixo (1-2) — score levemente positivo ainda é Negativo
    if rating <= 2:
        if score <= 0.3:
            return "Negativo"
        else:
            return "Neutro"  # score muito positivo mesmo com rating baixo

    # Rating 3 — neutro por natureza, confia no score NLP
    if rating == 3:
        if score > 0.15:
            return "Positivo"
        elif score < -0.15:
            return "Negativo"
        else:
            return "Neutro"

    return "Neutro"

# Aplica classificação híbrida
df["classificacao_sentimento"] = df.apply(classificar_hibrido, axis=1)

# --------------------------------------------------------
# Diagnóstico completo
# --------------------------------------------------------
print("=" * 55)
print("DISTRIBUIÇÃO DE SENTIMENTOS — Classificação Híbrida")
print("=" * 55)

dist_sent = df["classificacao_sentimento"].value_counts()
pct_sent  = (dist_sent / len(df) * 100).round(2)

diagnostico_sent = pd.DataFrame({
    "Quantidade": dist_sent,
    "% do Total": pct_sent
})
print(diagnostico_sent)

print("\n" + "=" * 55)
print("COERÊNCIA: RATING x SCORE x CLASSIFICAÇÃO")
print("=" * 55)

coerencia = df.groupby("rating").agg(
    score_medio    = ("sentimento_score", "mean"),
    pct_positivo   = ("classificacao_sentimento",
                      lambda x: (x == "Positivo").mean() * 100),
    pct_negativo   = ("classificacao_sentimento",
                      lambda x: (x == "Negativo").mean() * 100),
).round(2)

print(coerencia)
print("\n💡 Esperado: % Positivo cresce e % Negativo cai conforme rating sobe")

print("\n" + "=" * 55)
print("TESTE DOS CASOS CRÍTICOS CORRIGIDOS")
print("=" * 55)

casos_criticos = df[
    df["content"].str.contains(
        "não gostei|não vale|não recomendo|horrível|péssimo",
        case=False, na=False
    )
][["rating", "content", "sentimento_score",
   "classificacao_sentimento"]].head(8)

display(casos_criticos)

DISTRIBUIÇÃO DE SENTIMENTOS — Classificação Híbrida
                          Quantidade  % do Total
classificacao_sentimento                        
Positivo                      187190       92.31
Negativo                        9418        4.64
Neutro                          6177        3.05

COERÊNCIA: RATING x SCORE x CLASSIFICAÇÃO
        score_medio  pct_positivo  pct_negativo
rating                                         
1             -0.24          0.00         86.08
2             -0.09          0.00         78.36
3              0.18         37.31         18.42
4              0.64         95.27          0.00
5              0.84         99.16          0.00

💡 Esperado: % Positivo cresce e % Negativo cai conforme rating sobe

TESTE DOS CASOS CRÍTICOS CORRIGIDOS


,rating,content,sentimento_score,classificacao_sentimento
150,1,não gostei não.,-1.0,Negativo
421,1,não gostei uso essa marca mas não gostei este ...,-1.0,Negativo
505,1,não indico. não gostei produto não matiza o ca...,-1.0,Negativo
510,1,"não superou as expectativas,não gostei.",-1.0,Negativo
807,3,não gostei. já tive melhores e mais baratos.,-1.0,Negativo
811,5,"está vindo parece um gel,está diferente, estra...",-1.0,Neutro
856,2,não considero que valeu a pena. produto não co...,-1.0,Negativo
864,1,não vale nada tudo engano dinheiro jogado fora.,0.0,Negativo


In [17]:
# ============================================================
# Top palavras em reviews de 1⭐ vs 5⭐
# Um dos insights mais visuais do projeto no Power BI
# ============================================================

def top_palavras(dataframe, rating, n=20):
    """
    Retorna as N palavras mais frequentes
    para uma faixa de rating específica.
    """
    tokens_rating = dataframe[
        dataframe["rating"] == rating
    ]["tokens"].explode()

    contagem = Counter(tokens_rating.dropna())
    return pd.DataFrame(
        contagem.most_common(n),
        columns=["palavra", "frequencia"]
    )

# Gera para cada rating
for estrelas in [1, 2, 3, 4, 5]:
    top = top_palavras(df, estrelas, n=15)
    print(f"\n{'=' * 40}")
    print(f"⭐ TOP 15 PALAVRAS — RATING {estrelas}")
    print(f"{'=' * 40}")
    print(top.to_string(index=False))


⭐ TOP 15 PALAVRAS — RATING 1
 palavra  frequencia
     nao        6779
  cabelo        2488
    veio        1234
  cheiro         999
  gostei         931
    nada         673
      ja         589
    ruim         474
  parece         457
horrivel         435
   ficou         434
 pessimo         412
     bom         399
     nem         388
      so         382

⭐ TOP 15 PALAVRAS — RATING 2
  palavra  frequencia
      nao        2762
   cabelo        1103
   gostei         492
   cheiro         366
     veio         349
      bom         332
    achei         323
     nada         219
    ficou         209
      bem         205
      pra         184
       ja         180
resultado         176
     fica         168
   parece         164

⭐ TOP 15 PALAVRAS — RATING 3
  palavra  frequencia
      nao        4476
   cabelo        2061
      bom        1204
    achei         747
   cheiro         713
   gostei         680
      bem         555
     veio         482
      pra         422
  

In [18]:
# ============================================================
# Validação completa antes da exportação
# ============================================================

print("=" * 55)
print("RELATÓRIO DE VALIDAÇÃO — NLP")
print("=" * 55)

print(f"\n📊 SCORES")
print(f"  Score médio geral:     {df['sentimento_score'].mean():.4f}")
print(f"  Score reviews 1⭐:     {df[df['rating']==1]['sentimento_score'].mean():.4f}")
print(f"  Score reviews 5⭐:     {df[df['rating']==5]['sentimento_score'].mean():.4f}")

print(f"\n🏷️  CLASSIFICAÇÕES")
for classe in ["Positivo", "Neutro", "Negativo"]:
    qtd = (df["classificacao_sentimento"] == classe).sum()
    pct = qtd / len(df) * 100
    print(f"  {classe:10}: {qtd:,} ({pct:.2f}%)")

print(f"\n🔍 CASOS EXTREMOS")
print(f"\n  Reviews 5⭐ classificados como Negativo:")
casos = df[
    (df["rating"] == 5) &
    (df["classificacao_sentimento"] == "Negativo")
][["rating", "content", "sentimento_score"]].head(5)
display(casos)

print(f"\n  Reviews 1⭐ classificados como Positivo:")
casos2 = df[
    (df["rating"] == 1) &
    (df["classificacao_sentimento"] == "Positivo")
][["rating", "content", "sentimento_score"]].head(5)
display(casos2)

print(f"\n✅ Shape final: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"\n📋 Todas as colunas:")
for col in df.columns:
    print(f"   - {col}")

RELATÓRIO DE VALIDAÇÃO — NLP

📊 SCORES
  Score médio geral:     0.7603
  Score reviews 1⭐:     -0.2391
  Score reviews 5⭐:     0.8426

🏷️  CLASSIFICAÇÕES
  Positivo  : 187,190 (92.31%)
  Neutro    : 6,177 (3.05%)
  Negativo  : 9,418 (4.64%)

🔍 CASOS EXTREMOS

  Reviews 5⭐ classificados como Negativo:


,rating,content,sentimento_score



  Reviews 1⭐ classificados como Positivo:


,rating,content,sentimento_score



✅ Shape final: 202,785 linhas × 14 colunas

📋 Todas as colunas:
   - date
   - rating
   - content
   - product_url
   - qtd_palavras
   - review_curto
   - content_original
   - produto_id
   - slug
   - nome_produto
   - categoria
   - tokens
   - sentimento_score
   - classificacao_sentimento


In [19]:
# ============================================================
# Salva o dataset final com NLP para o Notebook 05
# ============================================================

# Remove coluna auxiliar de tokens (lista — não serve pro Power BI)
df_export = df.drop(columns=["tokens"])

caminho_saida = PROCESSED_PATH / "reviews_nlp.csv"
df_export.to_csv(caminho_saida, index=False, encoding="utf-8-sig")

print(f"✅ Dataset com NLP exportado com sucesso!")
print(f"📁 Caminho: {caminho_saida}")
print(f"📊 Shape final: {df_export.shape[0]:,} linhas × {df_export.shape[1]} colunas")
print(f"\n📋 Colunas exportadas:")
for col in df_export.columns:
    print(f"   - {col}")

✅ Dataset com NLP exportado com sucesso!
📁 Caminho: D:\GITHUB\portfolio-analista-dados\projetos\marketplace-review-intelligence\data\processed\reviews_nlp.csv
📊 Shape final: 202,785 linhas × 13 colunas

📋 Colunas exportadas:
   - date
   - rating
   - content
   - product_url
   - qtd_palavras
   - review_curto
   - content_original
   - produto_id
   - slug
   - nome_produto
   - categoria
   - sentimento_score
   - classificacao_sentimento
